# Experiment 10: Autoencoder in Deep Learning

**Objective**: Understand the architecture and training process of an Autoencoder, and demonstrate its ability to compress and reconstruct images.

**Dataset**: MNIST (Handwritten Digits)

**Methodology**:
1. Load and preprocess the MNIST dataset (we don't need the labels for this unsupervised task).
2. Build the Autoencoder model consisting of an **Encoder** and a **Decoder**.
3. Train the model to reconstruct its own input by minimizing the reconstruction loss.
4. Extract the trained Encoder to compress test images into a lower-dimensional latent space.
5. Visualize and compare the original inputs with the reconstructed outputs.

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")

## 2. Load and Preprocess Dataset
For Autoencoders, the target output is the input itself. Thus, we don't need `y_train` or `y_test`.

In [ ]:
# Load MNIST dataset
(X_train, _), (X_test, _) = keras.datasets.mnist.load_data()

# Normalize pixel values to be between 0 and 1
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Flatten the images (28x28 -> 784)
X_train = X_train.reshape((-1, 784))
X_test = X_test.reshape((-1, 784))

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

## 3. Define the Autoencoder Architecture
We will build a simple feed-forward neural network autoencoder. The architecture involves progressively reducing the dimensions (Encoder) until the bottleneck layer (Latent Space Representation), and then progressively expanding it back to the original dimension (Decoder).

In [ ]:
# This is the size of our encoded representations
encoding_dim = 32  # Flattens the input down to a vector of length 32 (compression factor of 784/32 = 24.5)

# This is our input image vector
input_img = keras.Input(shape=(784,))

# "encoded" is the encoded representation of the input (Bottleneck / Latent Space)
encoded = layers.Dense(128, activation='relu')(input_img)
encoded = layers.Dense(64, activation='relu')(encoded)
bottleneck = layers.Dense(encoding_dim, activation='relu')(encoded)

# "decoded" is the lossy reconstruction of the input
decoded = layers.Dense(64, activation='relu')(bottleneck)
decoded = layers.Dense(128, activation='relu')(decoded)
output_img = layers.Dense(784, activation='sigmoid')(decoded)

# The complete Autoencoder model mapping an input to its reconstruction
autoencoder = keras.Model(inputs=input_img, outputs=output_img, name="Autoencoder")

# Optional: We can also create separate Encoder and Decoder models for isolation
encoder = keras.Model(inputs=input_img, outputs=bottleneck, name="Encoder")

autoencoder.summary()

## 4. Compile and Train the Autoencoder
We use `binary_crossentropy` as our loss function (since inputs are scaled to [0,1] and the output layer uses a sigmoid activation function). The optimizer is `adam`.

In [ ]:
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

epochs = 20
batch_size = 256

print("Training the Autoencoder...")
history = autoencoder.fit(X_train, X_train,    # Input is X_train, Target is also X_train
                          epochs=epochs,
                          batch_size=batch_size,
                          shuffle=True,
                          validation_data=(X_test, X_test),
                          verbose=1)

## 5. Evaluate the Training Process

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Autoencoder Training Loss vs. Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss (Binary Crossentropy)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Visualize Original vs. Reconstructed Images
We use the trained autoencoder to predict (reconstruct) testing dataset examples.

In [ ]:
# Encode and decode some digits
# Note that we take them from the *test* set
encoded_imgs = encoder.predict(X_test)
decoded_imgs = autoencoder.predict(X_test)

n = 10  # Number of digits to display
plt.figure(figsize=(20, 6))

for i in range(n):
    # Display Original Images
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    plt.title("Original")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Display Encoded Representation (Latent Space Bottleneck)
    ax = plt.subplot(3, n, i + 1 + n)
    # The encoded image has shape (32,), we can reshape it to (8,4) or (4,8) to plot
    plt.imshow(encoded_imgs[i].reshape(8, 4), cmap='gray')
    plt.title("Encoded:\n(8x4 rep)")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Display Reconstructed Images
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap='gray')
    plt.title("Reconstructed")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

plt.tight_layout()
plt.show()

## Conclusion / Observations

1. **Unsupervised Learning Process**: Autoencoders demonstrate a self-supervised approach. By training the neural network mapping the dataset to itself ($X \rightarrow X$), we force the model to learn the intrinsic representations within the dataset without needing explicit classification labels.
2. **The Bottleneck Effect**: The bottleneck layer (in this experiment, possessing 32 neurons compared to the original 784 pixels) successfully extracts a lossy compressed latent representation of the number. Instead of memorizing the pixel indices, the autoencoder learns "what a number visually looks like" to recreate it effectively.
3. **Loss Function Behavior**: The `binary_crossentropy` smoothly decreases and converges alongside the validation loss, proving that the autoencoder can generalize pattern reconstruction properties to testing numbers it was not trained on.
4. **Visual Analysis**: The initial images are sharp but potentially contain noise or isolated details. The encoded representation effectively acts as an unrecognizable 8x4 latent code. Ultimately, the reconstructed output accurately yields the structural shape of the original image with a slight blurting effect due to the informational loss in the bottleneck section.